STATIC REVIEW ONLY — not a validated submission notebook. Use a fresh Kaggle Python 3.12/Linux x86_64 session with frozen source, competition data and a persisted wheelhouse attached. Offline bootstrap checks the full active dependency closure before installation. Both action switches default to False. No uploads or competition submission are performed. Canonical dataset IDs exclude .zarr. Frozen policies retain their empty-candidate failure.
For a rebuilt wheelhouse, paste the complete VERIFIED_WHEELHOUSE_LOCK block from notebook 01 into the bootstrap Cell. Both ZIP and manifest hashes identify the new artifact. Private Dataset wrappers are discovered recursively by manifest/hash, including extracted wheels or a ZIP-only mount. Multiple matching manifests/ZIPs stop as ambiguous.


In [ ]:
# NEW Kaggle Python 3.12/Linux x86_64 Cell, STATICALLY CHECKED ONLY.
# Run before importing numpy/torch/project modules in a fresh notebook kernel.
# No inference, tracking, solver, GT, upload, or competition submission.
ALLOW_OFFLINE_INSTALL = False  # Enable only when the user elects to restore packages.
BOOTSTRAP_PASS = False  # A failed rerun must not inherit a stale success flag.

import sys
import os
import platform
import hashlib
import json
import zipfile
import subprocess
from pathlib import Path
from email.parser import BytesParser
from importlib import metadata as md
from packaging.utils import canonicalize_name, parse_wheel_filename
from packaging.tags import sys_tags
from packaging.requirements import Requirement
from packaging.markers import default_environment
from packaging.specifiers import SpecifierSet

assert sys.platform == 'linux' and sys.version_info[:2] == (3, 12)
assert platform.machine() == 'x86_64'
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
SOURCE_SHA = '615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865'
# For a rebuild, replace this entire block with the VERIFIED_WHEELHOUSE_LOCK
# printed by notebook 01 after ZIP verification; persist that output with the ZIP.
# These defaults identify only the historical artifact, not an arbitrary rebuild.
WHEELHOUSE_LOCK = {
    'zip_sha256': 'b25787484b0a6430caaf203aa5557ffefd4e7713c00b537accaf47c4b0a1f1bc',
    'manifest_sha256': '9e132365f3c182a34d67a2c6b89d78f0bce246bf2bec6960d2f4ef824be3aafa',
}
WHEEL_MANIFEST_SHA = WHEELHOUSE_LOCK['manifest_sha256']
WHEEL_ZIP_SHA = WHEELHOUSE_LOCK['zip_sha256']
BASE_EXPECTED = {'torch': '2.10.0+cu128', 'numpy': '2.0.2', 'pandas': '2.3.3'}
PINNED = {'zarr':'3.1.6','numcodecs':'0.15.1','donfig':'0.8.1.post1',
          'geff':'1.3.0.1.2','geff-spec':'1.2.0','polars':'1.43.2',
          'polars-runtime-32':'1.43.2','bidict':'0.23.1','ilpy':'0.6.0',
          'pyscipopt':'6.2.1','rustworkx':'0.18.1','imagecodecs':'2026.3.6',
          'tracksdata':'0.1.0rc9.dev4+g7bfeaf845'}

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1048576), b''):
            h.update(b)
    return h.hexdigest()

def discover(filename):
    matches = []
    for directory, dirs, files in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in {'train', 'test', '.git'}
                   and not d.endswith(('.zarr', '.geff'))]
        if filename in files:
            matches.append(Path(directory) / filename)
    return sorted(matches)

sources = [p for p in discover('bundle_manifest.json') if file_sha(p) == SOURCE_SHA]
assert len(sources) == 1, ('SOURCE_MANIFEST_MATCHES', sources)
ROOT = sources[0].parent
source_manifest = json.loads(sources[0].read_text())
assert len(source_manifest['files']) == 29
for relative, record in source_manifest['files'].items():
    p = ROOT / relative
    assert p.is_file() and not p.is_symlink()
    assert p.stat().st_size == record['bytes'] and file_sha(p) == record['sha256'], relative
print('SOURCE_LOCKS_PASS', ROOT)

# Prefer the mounted extracted Dataset. ZIP-only input is verified then extracted
# into a NEW working directory; no source bundle is modified or repackaged.
manifests = [p for p in discover('wheelhouse_manifest.json') if file_sha(p) == WHEEL_MANIFEST_SHA]
assert len(manifests) <= 1, ('AMBIGUOUS_WHEELHOUSE', manifests)
if manifests:
    WHEELHOUSE = manifests[0].parent
else:
    zips = [p for p in discover('biohub_wheelhouse_py312_v1.zip') if file_sha(p) == WHEEL_ZIP_SHA]
    assert len(zips) == 1, 'WHEELHOUSE_NOT_PERSISTED_OR_WRONG_IDENTITY'
    WHEELHOUSE = Path('/kaggle/working/biohub_wheelhouse_verified_input')
    assert not WHEELHOUSE.exists(), 'Never overwrite extraction'
    with zipfile.ZipFile(zips[0]) as z:
        infos = z.infolist()
        assert len(infos) == len({i.filename for i in infos})
        raw = z.read('wheelhouse_manifest.json')
        assert hashlib.sha256(raw).hexdigest() == WHEEL_MANIFEST_SHA
        wm = json.loads(raw)
        expected = {'wheelhouse_manifest.json'} | {'wheels/' + r['filename'] for r in wm['wheels']}
        assert {i.filename for i in infos if not i.is_dir()} == expected
        for r in wm['wheels']:
            assert Path(r['filename']).name == r['filename'] and '\\' not in r['filename']
            content = z.read('wheels/' + r['filename'])
            assert len(content) == r['size_bytes'] and hashlib.sha256(content).hexdigest() == r['sha256']
        WHEELHOUSE.mkdir()
        for relative in sorted(expected):
            dest = WHEELHOUSE / relative
            dest.parent.mkdir(exist_ok=True)
            with dest.open('xb') as f:
                f.write(z.read(relative))

wm = json.loads((WHEELHOUSE / 'wheelhouse_manifest.json').read_text())
records = wm['wheels']
assert len(records) == 13 and len({r['filename'] for r in records}) == 13
assert {p.name for p in (WHEELHOUSE / 'wheels').iterdir()} == {r['filename'] for r in records}
overlay = {}
wheel_paths = []
target_tags = set(sys_tags())
for r in records:
    p = WHEELHOUSE / 'wheels' / r['filename']
    assert p.is_file() and not p.is_symlink()
    assert p.stat().st_size == r['size_bytes'] and file_sha(p) == r['sha256']
    _, _, _, tags = parse_wheel_filename(p.name)
    assert target_tags & tags, ('INCOMPATIBLE_WHEEL_TAG', p.name)
    with zipfile.ZipFile(p) as z:
        names = [n for n in z.namelist() if n.endswith('.dist-info/METADATA')]
        assert len(names) == 1
        m = BytesParser().parsebytes(z.read(names[0]))
    name = canonicalize_name(m['Name'])
    assert name not in overlay
    assert m['Version'] == PINNED[name]
    assert not m['Requires-Python'] or SpecifierSet(m['Requires-Python']).contains(platform.python_version())
    overlay[name] = (m['Version'], m.get_all('Requires-Dist', []))
    wheel_paths.append(str(p))
assert set(overlay) == set(PINNED)
for name, version in BASE_EXPECTED.items():
    assert md.version(name) == version, ('BASE_IMAGE_DRIFT', name, md.version(name))
print('WHEELHOUSE_FILES_AND_TAGS_PASS')

# Check recursive active Requires-Dist against the simulated overlay + base.
# Extras requested by a requirement propagate to the corresponding child.
# This is independent of pip check for unrelated preinstalled applications.
def closure_check(use_overlay):
    queue = [(name, frozenset()) for name in list(PINNED) + ['torch','numpy','pandas','scipy','packaging']]
    seen, versions, failures = set(), {}, []
    environment = default_environment()
    while queue:
        name, extras = queue.pop()
        name = canonicalize_name(name)
        key = (name, extras)
        if key in seen:
            continue
        seen.add(key)
        try:
            version, requirements = overlay[name] if use_overlay and name in overlay else (md.version(name), md.requires(name) or [])
        except md.PackageNotFoundError:
            failures.append([name, 'MISSING']); continue
        versions[name] = version
        for raw in requirements:
            try:
                req = Requirement(raw)
                active = req.marker is None or any(req.marker.evaluate({**environment, 'extra': e}) for e in ({''} | set(extras)))
                if not active:
                    continue
                child = canonicalize_name(req.name)
                child_version = overlay[child][0] if use_overlay and child in overlay else md.version(child)
                if req.url or not req.specifier.contains(child_version, prereleases=True):
                    failures.append([name, raw, child_version])
                queue.append((child, frozenset(req.extras)))
            except Exception as exc:
                failures.append([name, raw, type(exc).__name__, str(exc)])
    print('PROJECT_CLOSURE', json.dumps({'versions': versions, 'failures': failures}, sort_keys=True))
    assert not failures, 'PROJECT_CLOSURE_BLOCKED: add missing compatible wheels; do not patch versions blindly'
    return versions

closure_check(True)
assert ALLOW_OFFLINE_INSTALL, 'PREINSTALL_CHECKS_PASS; installation disabled by default'
already_loaded = [m for m in ('torch','numpy','pandas','polars','zarr','tracksdata','geff') if m in sys.modules]
assert not already_loaded, ('USE_FRESH_KERNEL_BEFORE_INSTALL', already_loaded)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--only-binary=:all:', '--find-links', str(WHEELHOUSE / 'wheels'), *wheel_paths], check=True)
for name, version in {**BASE_EXPECTED, **PINNED}.items():
    assert md.version(name) == version
versions = closure_check(False)
check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True)
print('GLOBAL_PIP_CHECK_EXIT', check.returncode, '\n', check.stdout, check.stderr)

for rel in reversed(['external/official/scripts','external/official/src','src','scripts']):
    sys.path.insert(0, str(ROOT / rel))
import importlib
project_modules = ['tracking_cellmot.io','tracking_cellmot.metrics','tracking_cellmot.models',
                   'train_unet_transformer_mps','predict_unet_transformer_mps',
                   'biohub_cell_tracking.frozen_v9','geffs_to_csv','csv_to_geffs','evaluate','dataspec','augmentations']
for name in ['torch','numpy','pandas','scipy','zarr','numcodecs','donfig','geff','geff_spec','tracksdata',
             'polars','bidict','ilpy','pyscipopt','rustworkx','imagecodecs'] + project_modules:
    module = importlib.import_module(name)
    if name in project_modules:
        origin = Path(module.__file__).resolve()
        assert origin.is_relative_to(ROOT.resolve())
        relative = origin.relative_to(ROOT.resolve()).as_posix()
        assert relative in source_manifest['files']
        assert file_sha(origin) == source_manifest['files'][relative]['sha256']
    print('IMPORT_PASS', name, getattr(module, '__file__', None))
import torch
assert torch.cuda.is_available(), 'CUDA_UNAVAILABLE_NO_CPU_FALLBACK'
print('CUDA_AVAILABLE', torch.cuda.get_device_name(0), torch.version.cuda)
report = Path('/kaggle/working/biohub_offline_environment_report.json')
with report.open('x') as f:
    json.dump({'source_manifest_sha256': SOURCE_SHA, 'wheel_manifest_sha256': WHEEL_MANIFEST_SHA, 'wheelhouse_lock': WHEELHOUSE_LOCK,
               'wheelhouse_path': str(WHEELHOUSE),
               'versions': versions, 'import_smoke': 'PASS', 'cuda_available': True,
               'pip_check_exit': check.returncode, 'pip_check_output': check.stdout,
               'inference_run': False, 'gt_read': False}, f, indent=2)
BOOTSTRAP_PASS = True
print('OFFLINE_INSTALL_PASS; PROJECT_IMPORT_SMOKE_PASS; CUDA_AVAILABILITY_PASS')
print('INFERENCE_RUN=NO; TRACKING_RUN=NO; EVALUATOR_RUN=NO')


In [ ]:
# NEW Kaggle Cell; run only after offline_bootstrap_cell.py succeeds.
# Full Kaggle Cell NOT RUN. Helpers CPU-tested using existing 14.14 artifacts;
# official conversion tested in memory. No model inference/tracker/GT execution.
# Changing this switch is an explicit decision to run both full CUDA predictions
# and the frozen tracker. This Cell does NOT upload or submit anything.
RUN_FROZEN_PIPELINE = False
assert RUN_FROZEN_PIPELINE, 'INFERENCE_AND_TRACKING_NOT_AUTHORIZED_BY_DEFAULT'
assert globals().get('BOOTSTRAP_PASS') is True

import csv
import gc
import itertools
import math
import shutil
from collections import Counter
from dataclasses import asdict
import numpy as np
import torch
import zarr
import predict_unet_transformer_mps as predictor
from biohub_cell_tracking import frozen_v9 as frozen
from tracking_cellmot.io import open_dataset, save_graph
from geffs_to_csv import geffs_to_csv
from csv_to_geffs import csv_to_geffs

WORK = Path('/kaggle/working/biohub_1415_recovery_v1')
FINAL = Path('/kaggle/working/submission.csv')
assert not WORK.exists() and not FINAL.exists(), 'Refuse to overwrite existing artifacts'
assert torch.cuda.is_available(), 'No silent CPU fallback'
for relative, r in source_manifest['files'].items():
    assert file_sha(ROOT / relative) == r['sha256'], relative
competition_roots = [Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
                     Path('/kaggle/input/biohub-cell-tracking-during-development')]
test_dirs = list(dict.fromkeys((p / 'test').resolve() for p in competition_roots if (p / 'test').is_dir()))
assert len(test_dirs) == 1, ('COMPETITION_MOUNT_AMBIGUOUS_OR_MISSING', test_dirs)
TEST_DIR = test_dirs[0]
zarrs = sorted(p for p in TEST_DIR.glob('*.zarr') if p.is_dir())
# predict() uses data_dir/name and writes name.geff; open_dataset resolves stem.zarr.
# frozen_v9 uses name.geff and open_dataset(TEST_DIR/name). Keep the same stem.
# Do not mix this new run with the historical <id>.zarr.geff naming convention.
names = [p.stem for p in zarrs]
assert names and len(names) == len(set(names))
assert all(Path(n).name == n and not n.endswith('.zarr') for n in names)
shapes, scales = {}, {}
for name, p in zip(names, zarrs, strict=True):
    ds = open_dataset(TEST_DIR / name, load_image=False, require_tracks=False)
    assert Path(ds.zarr_path).resolve() == p.resolve(), 'Official IO changed dataset name resolution'
    shapes[name] = tuple(int(v) for v in ds.image_shape)
    scales[name] = tuple(float(v) for v in ds.scale)
    assert len(shapes[name]) == 4 and all(v > 0 for v in shapes[name])
    assert len(scales[name]) == 3 and all(math.isfinite(v) and v > 0 for v in scales[name])
    assert '0.001' in ds.quantiles and '0.999' in ds.quantiles, 'Frozen predictor requires stored quantiles'
WORK.mkdir()
SPLITS = WORK / 'test_splits.json'
with SPLITS.open('x') as f:
    json.dump([{'test': names}], f, indent=2)
WEIGHTS = ROOT / 'models/official_baseline/official_baseline_fold0_warm2_lr1e5/split_0/edge_predictor_best.pth'

def tree_sha(path):
    path = Path(path)
    assert not path.is_symlink()
    if path.is_file():
        return file_sha(path)
    files = sorted(p for p in path.rglob('*') if p.is_file())
    assert files and not any(p.is_symlink() for p in path.rglob('*'))
    h = hashlib.sha256()
    for p in files:
        h.update(p.relative_to(path).as_posix().encode() + b'\0' + bytes.fromhex(file_sha(p)) + b'\0')
    return h.hexdigest()

def validate_graph(graph, name, *, prediction=False):
    nodes, edges = graph.node_attrs(), graph.edge_attrs()
    assert {'node_id','t','z','y','x'} <= set(nodes.columns)
    points = {}
    for row in nodes.select('node_id','t','z','y','x').iter_rows(named=True):
        raw_id = row['node_id']
        node_id = int(raw_id)
        assert node_id == raw_id and node_id >= 0 and node_id not in points
        v = tuple(float(row[k]) for k in ('t','z','y','x'))
        assert all(math.isfinite(x) and 0 <= x < limit for x, limit in zip(v, shapes[name], strict=True))
        assert v[0].is_integer()
        points[node_id] = v
    incoming, outgoing, pairs = Counter(), Counter(), set()
    for row in edges.iter_rows(named=True):
        s, t = int(row['source_id']), int(row['target_id'])
        assert s == row['source_id'] and t == row['target_id']
        assert s in points and t in points and s != t and (s,t) not in pairs
        dt = points[t][0] - points[s][0]
        assert dt > 0 and (not prediction or dt == 1)
        pairs.add((s,t)); incoming[t] += 1; outgoing[s] += 1
        if prediction:
            assert math.isfinite(float(row['edge_prob'])) and 0 <= row['edge_prob'] <= 1
            assert math.isfinite(float(row['edge_dist'])) and row['edge_dist'] >= 0
    assert max(incoming.values(), default=0) <= 1
    assert max(outgoing.values(), default=0) <= 2
    return {'nodes': len(points), 'edges': len(pairs)}

def verify_directory(directory, *, prediction=False):
    assert {p.name for p in directory.glob('*.geff')} == {n + '.geff' for n in names}
    result = {}
    for name in names:
        p = directory / (name + '.geff')
        graph = frozen.load_graph(p)
        result[name] = {**validate_graph(graph, name, prediction=prediction), 'tree_sha256': tree_sha(p)}
        del graph
    return result

prediction_dirs, prediction_records, prediction_configs = {}, {}, {}
for threshold, method in [(0.995, 'warm2_det0995'), (0.9975, 'warm2_det09975')]:
    destination = WORK / 'predictions' / 'kaggle_1415' / method / 'split_0'
    assert not destination.exists(), 'predict() deletes existing GEFFs; refuse rerun'
    cfg = predictor.PredictConfig(det_threshold=threshold, pool_kernel_um=3.0, use_ilp=False)
    assert cfg.det_tta and cfg.edge_activation == 'softmax' and cfg.threshold == 0.5
    assert cfg.max_parents_per_node == 1 and cfg.max_children_per_node == 2
    prediction_configs[method] = asdict(cfg)
    previous_path, previous_user = predictor.PREDICTIONS_PATH, predictor.USERNAME
    try:
        predictor.PREDICTIONS_PATH = WORK / 'predictions'
        predictor.USERNAME = 'kaggle_1415'
        predictor.predict(data_dir=TEST_DIR, fold=0, splits_file=SPLITS, weights_path=WEIGHTS,
                          cfg=cfg, method=method, debug_video=None, unet_batch_size=4,
                          video_slice=None, evaluate=False, max_frames=None)
    finally:
        predictor.PREDICTIONS_PATH, predictor.USERNAME = previous_path, previous_user
    prediction_dirs[threshold] = destination
    prediction_records[method] = verify_directory(destination, prediction=True)
    with (WORK / (method + '_validation.json')).open('x') as f:
        json.dump(prediction_records[method], f, indent=2)
    print('CUDA_FULL_INFERENCE_AND_GEFF_PASS', method)
    gc.collect()

cfg = frozen.FROZEN_V9_CONFIG
assert cfg.det_threshold == 0.995 and cfg.high_threshold == 0.9975
assert cfg.confidence_component_limit == 9 and tuple(cfg.model_downsample_zyx) == (1.,4.,4.)
# Intentionally preserve ValueError when a global candidate class is empty.
policies = frozen.build_frozen_v9_policies(names, prediction_dirs[0.995], prediction_dirs[0.9975],
                                         train_dir=TEST_DIR, prefix_map=None, config=cfg, progress=True)
policy_counts = {field: len(getattr(policies, field)) for field in
                 ('gap_candidates','single_policy','isolated_candidates','isolated_scope','strict_two_policy')}
print('POLICY_BUILD_PASS', policy_counts)
TRACK = WORK / 'tracking_geffs'
TRACK.mkdir()
applications, track_counts = {}, {}
for name in names:
    graph, scale, application = frozen.apply_frozen_v9(name, prediction_dirs[0.995], prediction_dirs[0.9975],
                                                      policies, train_dir=TEST_DIR, config=cfg,
                                                      strict_single_policy=True)
    assert np.array_equal(np.asarray(scale), np.asarray(scales[name]))
    track_counts[name] = validate_graph(graph, name)
    out = TRACK / (name + '.geff')
    assert not out.exists()
    save_graph(graph, out, overwrite=False)
    applications[name] = asdict(application)
    del graph
track_records = verify_directory(TRACK)
assert all(track_counts[n] == {k: track_records[n][k] for k in ('nodes','edges')} for n in names)
print('FROZEN_TRACKING_GEFF_PASS')

COLUMNS = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
STAGING = WORK / 'submission_unverified.csv'
assert not STAGING.exists()
geffs_to_csv(TRACK, STAGING)

def semantic_digest(values):
    h = hashlib.sha256()
    for value in sorted(values):
        h.update(json.dumps(value, separators=(',',':')).encode() + b'\n')
    return h.hexdigest()

def validate_csv(path):
    # Official exporter groups rows by dataset; keep one dataset in memory.
    results, seen, row_count = {}, set(), 0
    with path.open(newline='') as f:
        reader = csv.DictReader(f)
        assert reader.fieldnames == COLUMNS
        for name, group in itertools.groupby(reader, key=lambda r: r['dataset']):
            assert name in names and name not in seen, 'Foreign or non-contiguous dataset'
            seen.add(name)
            points, pairs = {}, []
            incoming, outgoing = Counter(), Counter()
            for row in group:
                assert None not in row and all(v is not None for v in row.values())
                values = {k: int(row[k]) for k in COLUMNS if k not in ('id','dataset','row_type')}
                # Official CSV importer ignores id entirely; no zero-origin restriction.
                row_count += 1
                if row['row_type'] == 'node':
                    nid = values['node_id']
                    assert nid >= 0 and nid not in points
                    assert values['source_id'] == values['target_id'] == -1
                    v = tuple(values[k] for k in ('t','z','y','x'))
                    # Official exporter rounds z/y/x with Polars. Do not round again
                    # or reject a valid pre-export boundary coordinate after rounding.
                    assert 0 <= v[0] < shapes[name][0]
                    points[nid] = v
                else:
                    assert row['row_type'] == 'edge'
                    assert all(values[k] == -1 for k in ('node_id','t','z','y','x'))
                    pairs.append((values['source_id'], values['target_id']))
            assert points, 'Dataset has no node rows; stop rather than fabricate'
            assert len(pairs) == len(set(pairs))
            signatures = []
            for s,t in pairs:
                assert s in points and t in points and points[s][0] < points[t][0]
                incoming[t] += 1; outgoing[s] += 1
                signatures.append((points[s], points[t]))
            assert max(incoming.values(), default=0) <= 1 and max(outgoing.values(), default=0) <= 2
            # Distinct nodes may share rounded coordinates. Digests retain multiplicity.
            # This compares coordinate/edge multisets, not graph isomorphism.
            assert len(points) == track_counts[name]['nodes'] and len(pairs) == track_counts[name]['edges']
            results[name] = {'nodes':len(points), 'edges':len(pairs),
                             'node_semantics_sha256':semantic_digest(points.values()),
                             'directed_edge_semantics_sha256':semantic_digest(signatures)}
    assert seen == set(names)
    assert row_count == sum(v['nodes'] + v['edges'] for v in results.values())
    return {'rows':row_count, 'datasets':results}

before = validate_csv(STAGING)
ROUNDTRIP = WORK / 'roundtrip_geffs'
ROUNDTRIP_CSV = WORK / 'roundtrip.csv'
assert not ROUNDTRIP.exists() and not ROUNDTRIP_CSV.exists()
csv_to_geffs(STAGING, ROUNDTRIP, overwrite=False)
assert {p.stem for p in ROUNDTRIP.glob('*.geff')} == set(names)
geffs_to_csv(ROUNDTRIP, ROUNDTRIP_CSV)
assert validate_csv(ROUNDTRIP_CSV) == before, 'Official roundtrip changed graph semantics'
for method, threshold in [('warm2_det0995',0.995),('warm2_det09975',0.9975)]:
    assert verify_directory(prediction_dirs[threshold], prediction=True) == prediction_records[method]
for relative, r in source_manifest['files'].items():
    assert file_sha(ROOT / relative) == r['sha256']
assert verify_directory(TRACK) == track_records
with STAGING.open('rb') as src, FINAL.open('xb') as dst:
    shutil.copyfileobj(src, dst)
assert file_sha(STAGING) == file_sha(FINAL)
with (WORK / 'run_manifest.json').open('x') as f:
    json.dump({'status':'CSV_AND_OFFICIAL_ROUNDTRIP_PASS', 'source_manifest_sha256':SOURCE_SHA,
               'wheel_manifest_sha256':WHEEL_MANIFEST_SHA, 'test_ids':names,
               'shapes':shapes, 'scales':scales, 'prediction_configs':prediction_configs,
               'frozen_config':asdict(cfg), 'predictions':prediction_records,
               'policy_counts':policy_counts, 'applications':applications, 'tracking':track_records,
               'csv_validation':before, 'submission_sha256':file_sha(FINAL),
               'submission_bytes':FINAL.stat().st_size, 'gt_read':False,
               'evaluator_run':False, 'competition_submission':False}, f, indent=2)
print('CSV_STRUCTURE_PASS; OFFICIAL_ROUNDTRIP_SEMANTICS_PASS')
print('SUBMISSION_FILE_READY', FINAL, FINAL.stat().st_size, file_sha(FINAL))
print('KAGGLE_VERSION_OUTPUT_VERIFIED=NO; COMPETITION_SUBMITTED=NO; GT_READ=NO')
